# Graphic design 01

Macro idea → designer’s proposal and image prompt → generated poster → same designer
examines the poster → revises its proposal or prompt → next poster.

The designer judges whether visible choices communicate the macro idea, using semiotic
reasoning where helpful. Its own proposal is provisional. No independent critic is required.


## 1. Setup

Launch Jupyter from the repository root or this experiment directory. Install the project with `python -m pip install -e ".[notebook]"` first.
Load local configuration without displaying secrets. Existing process variables take priority,
then `.env.local`, then `.env`. The generation cell runs a paid API call only when `RUN_GENERATION` is set to `True`.


In [10]:
import os
import json
from pathlib import Path
from dotenv import dotenv_values
from copy import deepcopy
from IPython.display import Markdown, display
from graphic_design_helper.workflow import generate_round, design_round, load_rounds
from graphic_design_helper.images import compare_images
from graphic_design_helper.records import save_experiment, start_run

# Locate the repository from either its root or an experiment directory.
REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src/graphic_design_helper").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Launch Jupyter inside the graphic-design-helper repository.")
EXPERIMENT_DIR = REPO_ROOT / "experiments" / "pilot1"
OUTPUT_DIR = EXPERIMENT_DIR / "outputs"

local_config = {
    **dotenv_values(REPO_ROOT / ".env"),
    **dotenv_values(REPO_ROOT / ".env.local"),
}
for key, value in local_config.items():
    if value is not None:
        os.environ.setdefault(key, value)
del local_config
from graphic_design_helper.prompt_builder import build_designer_request, request_token
from graphic_design_helper.workflow import review_token, compose_prompt

from graphic_design_helper.proposal_presentation import render_design_rationale


## 2. Design task and requirements

Specify the communication purpose, audience, viewing context, deliverable, and constraints
below. These values populate the corresponding sections of the designer input template.
The internal `brief` variable stores these requirements; no separate brief document is needed.
Optional additional wording can be supplied through `original_user_request`.

Use the experiment-selection cell below to start a new run or restore a saved one.
Restoring a run also restores its task wording and latest proposal. New run folders
are created only when a designer call is enabled.


In [11]:
original_user_request = None  # Preserve a separate user request verbatim when supplied.
brief = {
    "core_message": None,  # Unspecified: the designer must not treat an inference as confirmed.
    "topic": "Slowing down in everyday life",
    "purpose": "Invite people to reconsider pressure to remain continuously productive and allow a brief pause.",
    "audience": "Students or office workers feeling pressure to stay productive.",
    "setting": "A poster encountered briefly in a shared indoor space.",
    "tone": "Inviting rather than blaming or patronizing.",
    "constraints": "Avoid unsupported factual, health, or society-wide claims.",
    "deliverable": "A portrait poster in English. Propose the visible wording.",
    "exact_copy": None,  # Supply text here only if it is a genuine requirement.
}
previous_headline = "A moment, without a project."  # Not included in the request.
if "rounds" not in globals():
    rounds = []
if "proposal_records" not in globals():
    proposal_records = []

if "run_dir" not in globals():
    run_dir = None


### Select a new experiment or resume a saved one

Set `RESUME_RUN` to a folder name such as `"run_01"` to restore its latest completed
proposal without calling a model. Set it to `None` to prepare a new experiment;
the new directory is created only when the designer call is enabled.
Run this cell once when selecting an experiment. Running it again discards in-memory
drafts and reloads the saved state. After restarting the kernel, run Setup and this
section again. A page refresh alone does not reload Python modules or variables.

For a saved proposal awaiting its first image, leave `RUN_DESIGNER = False` and
continue to Section 5 to prepare, preview, and review the image request again.
For self-review of a generated poster, return to Section 3 and explicitly enable
the designer call after reviewing its input. Model-call switches default to `False`.


In [12]:
RESUME_RUN = None  # Start a new experiment.

if RESUME_RUN is not None:
    if not isinstance(RESUME_RUN, str) or Path(RESUME_RUN).name != RESUME_RUN:
        raise ValueError("Choose a run folder name directly under outputs.")
    restored_run = (OUTPUT_DIR / RESUME_RUN).resolve(strict=True)
    if restored_run.parent != OUTPUT_DIR.resolve():
        raise ValueError("The selected run must be directly under outputs.")
    manifest = json.loads((restored_run / "run.json").read_text(encoding="utf-8"))
    restored_input = json.loads((restored_run / "brief.json").read_text(encoding="utf-8"))
    restored_rounds = load_rounds(restored_run)
    restored_proposal = None
    if manifest.get("latest_designer_attempt"):
        attempt_dir = Path(manifest["latest_designer_attempt"]).resolve(strict=True)
        if not attempt_dir.is_relative_to(restored_run):
            raise ValueError("Saved proposal path is outside this run; repair its paths first.")
        restored_proposal = json.loads((attempt_dir / "response.json").read_text(encoding="utf-8"))
        if restored_proposal.get("status") != "completed":
            raise ValueError("The latest designer attempt is incomplete; inspect its record before continuing.")
        # Later attempts may include clarification added after the run began.
        restored_input = restored_proposal["designer_input"]
    run_dir = restored_run
    rounds = restored_rounds
    brief = deepcopy(restored_input["brief"])
    original_user_request = restored_input.get("original_user_request")
else:
    run_dir = None
    rounds = []
    restored_proposal = None

# Approval and preview state must be rebuilt from the selected experiment.
for stale_name in ("proposal_record", "image_spec", "designer_request", "active_round",
                   "image_prompt_preview", "image_preview_context", "designer_preview_token"):
    globals().pop(stale_name, None)
reviewed_token = None
proposal_records = []
if restored_proposal is not None:
    proposal_record = restored_proposal
    proposal_records.append(deepcopy(proposal_record))
    display(Markdown(render_design_rationale(proposal_record["proposal"])))
print(f"Experiment: {run_dir if run_dir is not None else 'new (not created yet)'}")
print(f"Saved image attempts: {len(rounds)}; proposal loaded: {restored_proposal is not None}")


Experiment: new (not created yet)
Saved image attempts: 0; proposal loaded: False


## 3. Preview the designer request

Initial proposals use `templates/designer-input-template.md`: the design task, audience,
requirements, design decisions, design approach, and proposal instructions.
Later rounds use `templates/designer-review-input-template.md`, which also receives the
previous proposal, the exact image prompt used, and the actual PNG as an image attachment.
The JSON response schema is shown separately because it is sent as the API's response
format. Editing instructions, schema, model settings, or the image requires a new preview.


In [13]:
designer_model = os.getenv("OPENAI_DESIGNER_MODEL", "").strip() or "gpt-5.6-luna"
designer_effort = "medium"

def current_designer_request():
    revision_context = None
    designer_image = None
    if rounds:
        previous = rounds[-1]
        if brief != previous["research"]["brief"]:
            raise ValueError("The macro brief changed. Start a new run for a new communication objective.")
        if not previous.get("image_path"):
            raise ValueError("The last attempt has no image. Resolve that attempt before continuing.")
        revision_context = {
            "parent_folder": previous["folder"],
            "previous_proposal": previous["research"].get("proposal"),
            "previous_image_spec": previous.get("image_spec"),
            "previous_prompt": previous.get("image_prompt") or previous.get("generation", {}).get("prompt") or previous.get("production_prompt"),
        }
        designer_image = previous["image_path"]
    return build_designer_request(
        brief, revision_context, original_user_request=original_user_request,
        image_path=designer_image, model=designer_model, reasoning_effort=designer_effort,
    )

if rounds and rounds[-1].get("status") in {"requested", "failed", "interrupted"}:
    designer_request = None
    designer_preview_token = None
    print("No image is available for self-review. For a failed/interrupted call, continue to Section 5 and explicitly retry in Section 6.")
else:
    designer_request = current_designer_request()
    designer_preview_token = request_token(designer_request)
    print(designer_request["prompt"])
    print(json.dumps({key: value for key, value in designer_request.items()
                      if key not in {"prompt", "designer_input"}}, ensure_ascii=False, indent=2))
    if designer_request.get("image_path"):
        display(compare_images([designer_request["image_path"]], ["Poster for self-review"]))


# Design task

## Topic

Slowing down in everyday life

## Communication purpose

Invite people to reconsider pressure to remain continuously productive and allow a brief pause.

## Core message

Not specified.

## Deliverable

A portrait poster in English. Propose the visible wording.

# Audience and viewing context

## Audience

Students or office workers feeling pressure to stay productive.

## Viewing context

A poster encountered briefly in a shared indoor space.

# Requirements and constraints

## Hard constraints

Avoid unsupported factual, health, or society-wide claims.

## Tone

Inviting rather than blaming or patronizing.

## Exact visible copy

Not specified.

# Design decisions

Develop the concept, signs, and intended readings for the communication purpose and viewing context, then select a supporting visual style and implement both through imagery, composition, typography, color, and wording. Respect all specified requirements, including exact visible copy when supplied.

## 4. Designer: propose, clarify, or review

Enable one paid designer call after reading the preview. No image call follows automatically.
If status is `needs_clarification` or `needs_sources`, supply the missing information
and return to the designer preview. `image_spec` remains null until the proposal is ready.
After self-review, inspect `revision_summary` and stop if no useful change is justified.
Each call is saved as a separate attempt, including failures and repeated proposals.


In [14]:
RUN_DESIGNER = True  # Set to False to skip the designer call and only preview the request.
if RUN_DESIGNER:
    if request_token(current_designer_request()) != designer_preview_token:
        raise ValueError("Inputs changed. Preview the designer request again.")
    if run_dir is None:
        run_dir = start_run(brief, original_user_request=original_user_request, output_dir=OUTPUT_DIR)
    proposal_record = design_round(
        designer_request, approved_token=designer_preview_token,
        run_dir=run_dir, history=rounds,
    )
    proposal_records.append(deepcopy(proposal_record))
    display(Markdown(render_design_rationale(proposal_record["proposal"])))
    print(f"Saved designer attempt: {proposal_record['folder']}")
else:
    print("Designer call is off. Preview the request, then enable when ready.")


# Design rationale

## Proposal status

ready

## 1. Communication purpose and context

<!-- Explain the purpose, audience, and viewing conditions relevant to this design. -->
Create a quickly readable indoor poster that gently interrupts the visual language of constant tasks and gives students or office workers permission to take a brief pause.

## 2. Core design concept

<!-- Introduce the overall idea and its principal elements before analyzing details. -->
A structured field of small task-like marks is interrupted by a calm, generous opening. The opening becomes the visual and verbal pause: a moment that is intentionally left unfilled rather than treated as wasted space.

## 3. Signs and intended readings

<!-- Use a subsection for each important element. Identify its visible form and
referent, then explain each relevant symbolic, iconic, or indexical relationship
separately: grounds, intended reading, and limitations. Do not force all three
categories. Explain how the elements work together; do not claim audience effects
as observed facts before they have been tested. -->
### Task marks and structured grid
- **Referent:** Everyday obligations, schedules, messages, assignments, and work tasks.
- **Relationship:** Primarily symbolic, because ordered blocks, check marks, and grid structures conventionally suggest organization and productivity. Their repeated density can also create an experiential suggestion of pressure, though that reading depends on context.
- **Grounds:** Students and office workers are familiar with lists, calendars, interfaces, and organized work surfaces.
- **Intended reading:** The poster acknowledges the familiar pressure to keep moving without depicting it as a personal failure.

### The open interruption
- **Referent:** A brief pause or moment not filled by a task.
- **Relationship:** Iconic and symbolic. The visible blank interval resembles an actual gap in a sequence, while the absence of marks conventionally communicates room, stopping, or release.
- **Grounds:** The interruption is made perceptible by contrast with the surrounding ordered marks and by its larger scale.
- **Intended reading:** There is legitimate room to stop briefly. The pause is part of the composition, not a defect in it.

### Soft circular form within the gap
- **Referent:** A contained minute of breathing room or attention.
- **Relationship:** Iconic by suggesting a simple clock or cycle without showing a precise time; symbolic because a circle can conventionally suggest continuity and completeness. It should not be read as a medical or wellness claim.
- **Grounds:** Its quiet, centered placement contrasts with the task grid and creates a visual resting point.
- **Intended reading:** The viewer is invited to settle their attention briefly.

### Wording: “MAKE ROOM / FOR A PAUSE”
- **Referent:** An invitation to allow a short interruption in activity.
- **Relationship:** Symbolic through ordinary English language, reinforced visually by the literal open space around the words.
- **Grounds:** The imperative is gentle rather than accusatory, and “make room” connects directly to the composition’s empty interval.
- **Intended reading:** The viewer may pause without needing to justify or complete anything first.

### Secondary wording: “Take one quiet minute.”
- **Referent:** A small, achievable action in the present moment.
- **Relationship:** Symbolic, with “minute” also supported by the circular time-like form. The exact duration is a proposed invitation, not a factual or health-related claim.
- **Grounds:** Its short length supports rapid reading in a shared indoor space.
- **Intended reading:** The request feels practical, bounded, and non-demanding.

## 4. Why this concept fits the task

<!-- Connect the concept to the communication purpose, audience, and constraints. -->
The concept uses a familiar productivity structure but does not attack work, ambition, or the viewer. By making the pause visually necessary to the composition, it communicates permission through form rather than through a lecture. A strong hierarchy and limited copy support brief encounters in corridors, break areas, classrooms, or offices. The wording is inviting and avoids unsupported claims about health, performance, or society.

## 5. Visual style and art direction

<!-- Derive the proposed style from the preceding signs and intended readings.
Explain which relationships its visual qualities support and how. State the direction,
its defining visual qualities, and what may vary. Distinguish user-specified,
proposed for review, and explicitly confirmed directions, citing supplied evidence
for confirmation. A historical style label is optional; concrete qualities are not. -->
### Style or reference direction
Proposed for review: a restrained contemporary editorial poster using a precise graphic grid, generous negative space, warm paper-like background, charcoal typography, and one muted accent color. No user-specified style or reference was supplied, and no historical movement is being claimed.

### How it supports the signs
The precise grid supports the symbolic language of tasks and productivity. Its deliberate interruption makes the open gap legible as a pause rather than an accidental omission. Flat color, limited detail, and clear typography preserve rapid recognition from a distance. The soft circular form introduces calm without relying on stereotyped wellness imagery.

### Defining visual qualities
- Portrait format with a strong vertical reading path.
- Warm off-white background and near-black text.
- Repeated small rectangular task marks arranged in an orderly field.
- One large, unmistakable quiet opening through the field.
- A muted blue-green circular form and a restrained coral accent used sparingly.
- Bold sans-serif headline, regular sans-serif supporting line, and generous line spacing.
- Slightly tactile paper grain, but no photographic realism or decorative clutter.

### Allowed variation and boundaries
The density and exact geometry of the task marks may vary, and the circular pause form may be slightly organic or precisely geometric. The central relationship must remain: an ordered field is visibly interrupted by a calm, usable opening. The headline must remain dominant, the palette must remain restrained, and the design must not become frantic, comedic, or overly therapeutic.

### Confirmation status
The visual direction is a designer proposal for review. The supplied brief confirms only the topic, purpose, audience, viewing context, English portrait format, and avoidance of unsupported claims. No style has been explicitly confirmed by the user.

## 6. Visual decisions and their implementation

<!-- Explain composition, typography, color, imagery, and hierarchy where relevant.
Connect each concrete choice to its sign relationship, the supporting style quality,
and the image instruction that implements it. Treat supplied style requirements as
constraints from the outset; revisit signs and style together if they conflict. Keep the
explanation understandable without opening the image prompt or machine records. -->
- **Composition:** The dense task field establishes the productivity referent; the large central opening preserves the intended interruption relationship. In the image specification, the field occupies the upper and side regions while the opening creates a clear vertical pause near the center.
- **Typography:** The headline is large and short for rapid indoor viewing. Its placement across or beside the opening makes “make room” spatially literal. This supports the symbolic wording and the style’s editorial clarity.
- **Visible copy:** The exact proposed lines are short enough to scan quickly and maintain the inviting tone. The wording must not be paraphrased or supplemented with slogans.
- **Visual treatment:** Flat, restrained marks preserve the grid-versus-gap relationship. The circular form supports the minute/pause association without becoming a literal clock or health diagram.
- **Color:** Charcoal and warm off-white maximize legibility; the muted blue-green identifies the pause as calm, while the small coral accent gently marks the interruption. Color is supportive rather than assigned a universal meaning.
- **Variation:** Mark density, circle geometry, and subtle paper texture may change, but the open interruption, visual hierarchy, wording, and non-judgmental tone must survive every variation.
- **Exclusions:** Avoid stressed faces, exhausted workers, phones overflowing with notifications, clocks showing urgency, medical imagery, productivity statistics, and imagery that frames pausing as a cure or obligation.

## 7. Alternatives and tradeoffs

- A more literal alternative could show a checklist with one blank line enlarged into a resting space; this would be faster to decode but less distinctive and more tied to office stationery.
- A more poetic alternative could use a continuous line that stops and becomes a quiet circle; this would feel calmer but might communicate “pause” less immediately in a brief public encounter.

## 8. Assumptions and possible misreadings

### Assumptions behind the proposal

- The poster may use a proposed headline and supporting line because exact visible copy was not supplied.
- A one-minute invitation is acceptable as a bounded, non-factual prompt rather than a claim about what the viewer needs.
- The poster will be viewed at approximately walking distance in a shared indoor environment.

### Uncertainties and possible misreadings

- Some viewers may read the grid as digital interface elements rather than general tasks; the headline and clear central gap should anchor the intended reading.
- The circular form may be read as a clock, target, or decorative shape; keeping it simple and placing it beside “one quiet minute” should reduce ambiguity.
- The imperative “MAKE ROOM” may still feel directive to some viewers, though the wording avoids blame and the surrounding negative space softens it.

## 9. How to review the result

<!-- Distinguish visible execution checks from questions requiring audience feedback. -->
- The headline is legible within a brief glance and remains the first or second element noticed.
- The ordered task field and the central opening are immediately distinguishable.
- The open area reads as intentional breathing room rather than missing artwork.
- The tone feels inviting, not blaming, patronizing, or productivity-oriented.
- The exact English copy is reproduced with the specified line breaks.
- No unsupported factual, health, or society-wide claim appears.
- The poster remains visually calm despite using a productivity-related motif.

## 10. Review observations and changes

<!-- For an initial proposal, state that no generated result has been reviewed.
For a revision, connect observations to the intended design and explain changes. -->
No explanation supplied.

## 11. Information needed to proceed

### Required sources

None listed.

### Questions to resolve

None listed.


Saved designer attempt: D:\DH-Research\graphic-design-helper\experiments\pilot1\outputs\run_01\rounds\01\designer\attempt_01


## 5. Review two deliverables: design rationale and image prompt

Read the design rationale in order: purpose and concept, signs and intended readings,
why the concept fits, the style that supports those relationships, and concrete visual
implementation. Review uncertainties and distinguish visible checks from audience feedback.
User-specified styles constrain sign selection from the outset; signs and style may
be revised together. When changing either, update the explanation and image instructions
together so the intended relationships remain explicit.
It is saved as `design-rationale.md` alongside the designer's `proposal.json`.

The editable image specification below is the second deliverable. Its assembled
prompt is what the image model receives; the design explanation is not sent to it.
You may edit `image_spec` without changing the saved model proposal. If an edit changes
the concept or its sign relationships, revisit the design explanation before approval.
Rerun the separate preview cell after edits. Reinitializing the draft discards unsaved edits.


In [15]:
image_spec = deepcopy(proposal_record["proposal"]["image_spec"]) if "proposal_record" in globals() else None
image_settings = {
    "model": os.getenv("OPENAI_IMAGE_MODEL", "gpt-image-2"),
    "size": "1024x1536", "quality": "medium",
}
pending_image = rounds[-1] if rounds and rounds[-1].get("status") in {"requested", "failed", "interrupted"} else None
if pending_image is not None:
    image_spec = deepcopy(pending_image["image_spec"])
    image_settings = deepcopy(pending_image["settings"])
reviewed_token = None

def current_research():
    if "proposal_record" not in globals():
        raise ValueError("Request a design proposal first.")
    inputs = proposal_record["designer_input"]
    if brief != inputs["brief"] or original_user_request != inputs["original_user_request"]:
        raise ValueError("The user input changed. Request a new proposal.")
    if pending_image is not None:
        return deepcopy(pending_image["research"])
    return deepcopy({"brief": brief, "original_user_request": original_user_request,
                     "proposal": proposal_record["proposal"], "designer_record": proposal_record})

print(json.dumps(image_spec, ensure_ascii=False, indent=2))


{
  "communication_objective": "Invite students and office workers to reconsider continuous productivity and allow themselves a brief, quiet pause.",
  "audience_and_context": "Students or office workers encountering a portrait poster briefly in a shared indoor space such as a hallway, common room, classroom, or office area.",
  "visible_copy": [
    "MAKE ROOM\nFOR A PAUSE",
    "Take one quiet minute."
  ],
  "composition": "Portrait poster with a warm off-white background. Build an orderly field of repeated small charcoal rectangular task marks and sparse check-like blocks across the upper half and side margins. Interrupt this field with one large, clean vertical opening centered slightly below the headline; the opening must be clearly intentional and remain mostly unfilled. Place the headline prominently across the upper-middle area, aligned with the opening. Place a simple muted blue-green circular form inside the quiet opening below the headline, with a small coral accent mark ne

In [16]:
# Rerun this cell after editing image_spec or the image prompt templates.
image_prompt_preview = compose_prompt(image_spec) if image_spec is not None else None
image_preview_context = request_token([image_spec, current_research(), image_settings]) if image_spec is not None else None
display(Markdown("# Image generation prompt\n\n" + image_prompt_preview)) if image_prompt_preview is not None else print("No ready image specification. Review the design rationale first.")
print(image_settings)


# Image generation prompt

# Rendering instructions

# Image rendering instructions

Render the poster described below. Only the strings in Exact visible text are intended
to appear in the artwork. Preserve their wording, punctuation, language, and intentional
line breaks. The JSON quotes and list syntax are delimiters, not artwork text.
An empty list means no visible text. Do not print instructions, section labels, or the
communication objective. Follow the composition, typography, visual treatment,
allowed variation, and exclusions. These instructions do not prescribe a default style.

Implement the direction described in Visual style and treatment together with the
composition and typography. Use the specified visual qualities to interpret any
style name; do not add stereotypical motifs, colors, or decoration merely because
a movement is named. Stay within Allowed variation and preserve the stated visual
character. Style labels and art-direction descriptions are instructions, not artwork text.

Preserve the specified elements and their spatial and hierarchical relationships
when applying the style. A style label does not authorize replacing those elements,
rearranging the message, or adding motifs. Use only the stated variation; execute
the supplied design rather than inventing a new concept to match a style.


# Communication objective

Invite students and office workers to reconsider continuous productivity and allow themselves a brief, quiet pause.

# Intended audience and viewing context

Students or office workers encountering a portrait poster briefly in a shared indoor space such as a hallway, common room, classroom, or office area.

# Exact visible text

[
  "MAKE ROOM\nFOR A PAUSE",
  "Take one quiet minute."
]

# Visual composition

<!-- Specify the elements and relationships that the style must preserve. -->
Portrait poster with a warm off-white background. Build an orderly field of repeated small charcoal rectangular task marks and sparse check-like blocks across the upper half and side margins. Interrupt this field with one large, clean vertical opening centered slightly below the headline; the opening must be clearly intentional and remain mostly unfilled. Place the headline prominently across the upper-middle area, aligned with the opening. Place a simple muted blue-green circular form inside the quiet opening below the headline, with a small coral accent mark near the break in the task field. Keep generous negative space around the supporting line near the lower third. Use a clear vertical hierarchy: headline, circular pause form, supporting line, then subtle task marks.

# Typography

Use a bold, highly legible contemporary sans-serif in near-black charcoal. Set “MAKE ROOM” on the first line and “FOR A PAUSE” on the second line, large uppercase, with tight but readable leading. Set “Take one quiet minute.” below the central form in a smaller regular-weight sans-serif. Maintain strong contrast, generous margins, and no additional text. Do not use decorative script, condensed display lettering, or distressed type.

# Visual style and treatment

Restrained contemporary editorial graphic poster, not photographic. Use flat geometric task marks, a simple soft-edged or subtly organic blue-green circle, and one small muted coral interruption. Palette: warm off-white paper, near-black charcoal, muted blue-green, and restrained dusty coral. Add very subtle paper grain only. The task marks should feel orderly and repetitive; the central opening should feel calm, spacious, and visually necessary. Keep the image quiet, precise, and inviting, with no literal people, no faces, no phones, no medical or wellness symbols, and no urgent clock imagery.

# Allowed variation

[
  "Vary the density and exact spacing of the small task marks while keeping them orderly.",
  "Allow the central circle to be either precisely geometric or slightly organic, but keep it simple and calm.",
  "Allow very subtle paper texture and minor registration softness.",
  "Preserve the portrait format, exact visible copy, strong headline hierarchy, central open interruption, restrained palette, and generous negative space."
]

# Exclusions

[
  "No additional words, captions, logos, statistics, claims, or decorative lettering.",
  "No blaming, scolding, or patronizing tone.",
  "No photographic realism, exhausted workers, stressed faces, overflowing notifications, alarm clocks, medical imagery, or productivity charts.",
  "Do not fill the central opening with extra tasks or ornament.",
  "Do not use bright neon colors, chaotic collage, aggressive diagonal motion, or dense visual clutter."
]


{'model': 'gpt-image-2', 'size': '1024x1536', 'quality': 'medium'}


In [17]:
MARK_PROMPT_REVIEWED = True
if MARK_PROMPT_REVIEWED:
    if image_prompt_preview is None:
        raise ValueError("Resolve clarification or sources, then preview a ready specification.")
    if request_token([image_spec, current_research(), image_settings]) != image_preview_context:
        raise ValueError("Draft, rationale, or settings changed. Preview again.")
    reviewed_token = review_token(image_spec, current_research(), image_settings,
                                  image_prompt=image_prompt_preview)
    print("Complete image request and research marked reviewed.")


Complete image request and research marked reviewed.


## 6. Generate the poster

Enable one generation after inspecting the proposed prompt. This experiment retains the
limit of three image calls in total, including failed calls and explicit retries. Each request saves its proposal, prompt and image.
The next poster is generated from the updated text prompt; this notebook does not edit the
previous image's pixels. The previous image is visual input to the designer's self-review.

If a call is interrupted, its records say `interrupted` with an unknown remote outcome.
No retry occurs automatically. Restore the run in Section 2, then use Section 5 to
preview and review the saved request. Set `RETRY_IMAGE = True`, enter `RETRY_REASON`,
and enable `RUN_GENERATION` to send one new call in the same round. A retry preserves
the earlier attempt and uses a new `attempt_<number>` directory. The request must
remain unchanged, and successful rounds cannot be retried through this switch.
A record still marked `requested` after a kernel crash needs inspection before it
can be classified as interrupted; it is never retried automatically.


In [ ]:
RUN_GENERATION = True
RETRY_IMAGE = False
RETRY_REASON = ""

if RUN_GENERATION:
    if reviewed_token is None:
        raise ValueError("Run the image prompt preview and MARK_PROMPT_REVIEWED cell first.")
    research = current_research()
    revision_decision = rounds[-1]["revision"] if RETRY_IMAGE and rounds else {
        "reason": research["proposal"]["revision_summary"] if rounds else "Initial proposal",
    }
    print("Waiting for the image response. Interrupting stops local waiting; the remote outcome may remain unknown.", flush=True)
    try:
        active_round = generate_round(
            image_spec, research, image_settings,
            approved_token=reviewed_token, history=rounds, revision=revision_decision,
            output_dir=OUTPUT_DIR, retry=RETRY_IMAGE, retry_reason=RETRY_REASON,
        )
        print(f"Saved round {active_round['round']} to {active_round['folder']}")
    finally:
        reviewed_token = None
        RUN_GENERATION = False
        RETRY_IMAGE = False
else:
    print("Image generation is off.")


In [ ]:
image_paths = [r["image_path"] for r in rounds if r.get("image_path")]
labels = [f"Round {r['round']}" for r in rounds if r.get("image_path")]
if image_paths:
    display(compare_images(image_paths, labels))


## 7. Look, reconsider, repeat

Return to Section 3 for self-review of the latest poster against the original brief.
Keep successful choices and stop when no useful change remains. The limit is three
image calls, including failed or uncertain calls and explicit retries; reaching it does not establish success.
A model's reading is a design judgment, not evidence of audience reception.

New records are grouped under `outputs/run_<number>/rounds/<number>/`. Each `designer/`
and `image/` directory contains separate `attempt_01`, `attempt_02`, ... folders with `request.json`,
`prompt.md`, and `response.json`. Designer attempts save `proposal.json`; image attempts
save the reviewed `image-spec.json` and `image.png`. The original proposal is preserved.

Use the experiment-selection cell in Section 2 after a restart. Historical outputs remain unchanged and
can be inspected directly; do not rerun them just to adopt the new record structure.


## 8. Optional audience readings

Before explaining the intention, ask: What did you notice first? What do you think this is saying?
Who seems to be speaking? Does it invite you to do anything? Record unexpected readings too.
These are exploratory observations, not evidence of long-term behavior change.

Store participants' words separately from your interpretation. Avoid identifying information.

For each element, compare the intended relationship with the reading people actually describe.
Ask about the imagery and arrangement in ordinary language; viewers need not know Peircean terms.
If something looks like evidence, ask what they think it documents. Do not supply that answer first.


In [ ]:
observations = []  # For each response: variant, anonymous label, verbatim wording, researcher interpretation.
reflection = {
    "what_worked": "",
    "unexpected_readings": "",
    "design_changes_to_try": "",
    "semiotic_assumptions_to_revisit": "",
    "relations_supported_or_challenged_by_responses": "",
    "source_or_evidence_confusions": "",
    "limitations": "No audience responses recorded in this reflection.",
}


## 9. Save a snapshot

Set `SAVE_SNAPSHOT` to `True` when you want to save the current state. Each save creates a new `outputs/snapshot_<number>/` folder
with notes, image copies, and hashes. Include only non-secret settings; never pass environment
variables, API keys, or client objects. The saved status distinguishes planning from collected evidence.


In [ ]:
SAVE_SNAPSHOT = False
if SAVE_SNAPSHOT:
    run = save_experiment({
        "status": "exploration" if rounds else "planning",
        "brief": brief, "designer_request": designer_request,
        "proposal_records": proposal_records,
        "image_spec_draft": image_spec, "rounds": rounds,
        "audience_observations": observations, "reflection": reflection,
    }, [r["image_path"] for r in rounds if r.get("image_path")], OUTPUT_DIR)
    print(f"Saved: {run}")
